In [21]:
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import torch
from PIL import Image, UnidentifiedImageError
from transformers import CLIPProcessor, CLIPModel

In [43]:
class CLIPEvaluator:
    def __init__(
        self,
        model_name: str = "openai/clip-vit-base-patch32",
        device: Optional[str] = None,
    ) -> None:
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model.eval()

    def build_quality_prompts(self) -> Tuple[List[str], List[str]]:
        positive_prompts = [
            "a high quality image",
            "a detailed coherent image",
            "a visually pleasing image",
            "a clear image",
            "a sharp natural image",
        ]

        negative_prompts = [
            "a low quality distorted image",
            "an unrealistic malformed image",
            "a blurry incoherent image",
            "an image with visible artifacts",
        ]

        return positive_prompts, negative_prompts

    @torch.no_grad()
    def score_image(
        self,
        image: Image.Image,
        positive_prompts: List[str],
        negative_prompts: List[str],
    ) -> Dict[str, object]:
        """
        Score an image using both positive and negative prompts.

        Final score:
            mean(positive prompt similarities) - mean(negative prompt similarities)

        Also returns each individual prompt score.
        """
        if len(negative_prompts) == 0:
            raise ValueError("negative_prompts must be non-empty")

        prompts = positive_prompts + negative_prompts

        inputs = self.processor(
            text=prompts,
            images=image,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        outputs = self.model(**inputs)

        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        similarities = (image_embeds @ text_embeds.T).squeeze(0).cpu()

        pos_sims = similarities[: len(positive_prompts)]
        neg_sims = similarities[len(positive_prompts):]

        positive_scores_list = pos_sims.tolist()
        negative_scores_list = neg_sims.tolist()

        positive_scores_dict = {
            p: float(s) for p, s in zip(positive_prompts, positive_scores_list)
        }
        negative_scores_dict = {
            p: float(s) for p, s in zip(negative_prompts, negative_scores_list)
        }

    

        return {

            "positive_scores_list": positive_scores_list,
            "negative_scores_list": negative_scores_list,
        }


def get_results(
    record: Dict[str, object],
    clip_evaluator: CLIPEvaluator,
) -> Dict[str, object]:
    """
    Build one output row from a record yielded by iter_dataset_2.
    """
    img_path = Path(record["path"])

    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    try:
        image = Image.open(img_path).convert("RGB")
    except UnidentifiedImageError as exc:
        raise ValueError(f"Could not read image: {img_path}") from exc

    positive_prompts, negative_prompts = clip_evaluator.build_quality_prompts()

    clip_result = clip_evaluator.score_image(
        image=image,
        positive_prompts=positive_prompts,
        negative_prompts=negative_prompts,
    )

    row = {
        "category": record["category"],
        "difficulty": record["difficulty"],
        "set_id": record["set_id"],
        "label": record["label"],
        "positive_scores_list": clip_result["positive_scores_list"],
        "negative_scores_list": clip_result["negative_scores_list"]
    }


    return row

In [44]:
GENERIC_TYPES = ["category_1", "category_2", "category_3"]
DIFFICULTY = [1,2,3]
LABELS = [1, 2, 3, 4, 5]
IMG_EXTS = [".png", ".jpg", ".jpeg"]


SET_RE = re.compile(r"^set(?P<id>\d+)$")

def find_label_image(folder: Path, label: int) -> Optional[Path]:
    """Return the first existing image file for a given label in folder."""
    for ext in IMG_EXTS:
        p = folder / f"{label}{ext}"
        if p.exists():
            return p
    return None

In [45]:
def iter_dataset_2(root: Path) -> Iterable[dict]:
    """
    Iterate through dataset/category_i/difficulty_j/setN/<label>.(png|jpg|...)
    """
    for category in GENERIC_TYPES:
        for difficulty in DIFFICULTY:
            base = root / category / f"difficulty_{difficulty}"

            if not base.exists():
                continue

            set_dirs = []
            for d in base.iterdir():
                if d.is_dir():
                    m = SET_RE.match(d.name)
                    if m:
                        set_dirs.append((int(m.group("id")), d))

            for set_id, set_dir in sorted(set_dirs, key=lambda x: x[0]):
                for label in LABELS:
                    p = find_label_image(set_dir, label)
                    if p:
                        yield {
                            "category": category,
                            "difficulty": difficulty,
                            "set_id": set_id,
                            "label": label,
                            "path": p,
                        }


In [ ]:
root_folder = ""

rows = []
clip_evaluator = CLIPEvaluator()

for record in iter_dataset_2(Path(root_folder)):
    row = get_results(
        record=record,
        clip_evaluator=clip_evaluator,
    )

    rows.append(row)

In [47]:
df = pd.DataFrame(rows)
df.to_csv("clip_quality.csv", index=False)

In [ ]:
# Example usage of CLIPEvaluator on individual images
image1 = Image.open(path_to_image1).convert("RGB")


best_positive_prompts = [
    "a high quality image",
    "a visually pleasing image",
    "a clear image",
    "a sharp natural image",
]

best_negative_prompts = [
    "an unrealistic malformed image",
    "a blurry incoherent image",
    "an image with visible artifacts",
]
clip_evaluator.score_image(
        image=image1,
        positive_prompts=best_positive_prompts,
        negative_prompts=best_negative_prompts,
    )
